<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/18-post-training-alignment.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Post-Training and Alignment** {#post-training-alignment}

Pretraining gives a model broad predictive competence, but next-token likelihood alone does not specify how a deployed assistant should follow instructions, refuse unsafe requests, express uncertainty, or trade helpfulness against verbosity. **Post-training** changes the behavior of a pretrained model using demonstrations, comparisons, reward signals, and targeted evaluation. **Alignment** is the broader engineering problem of making that behavior reliably reflect intended objectives and constraints under both ordinary and adversarial use.

![A staged post-training pipeline transforms a predictive base model into an evaluated deployment policy.](assets/dl18-post-training-pipeline.svg){fig-align="center" width="78%" fig-alt="A pipeline moves from a pretrained model through supervised fine-tuning and preference optimization to an evaluated policy, with evaluation feeding new data back into the process."}

*Original teaching diagram synthesized from the post-training pipeline in [InstructGPT](https://arxiv.org/abs/2203.02155), the [DPO paper](https://arxiv.org/abs/2305.18290), and [Constitutional AI](https://arxiv.org/abs/2212.08073).*

The chapter uses [NVIDIA HelpSteer2](https://huggingface.co/datasets/nvidia/HelpSteer2), introduced in the [HelpSteer2 paper](https://arxiv.org/abs/2406.08673). Its responses are rated for helpfulness, correctness, coherence, complexity, and verbosity. The repository stores a deterministic 240-prompt teaching subset of the official training split under the dataset's **CC BY 4.0** license. Every prompt has two responses; the higher weighted human score becomes $y_w$ and the other becomes $y_l$. This reduction supports reproducible CPU demonstrations, not a leaderboard claim.

<details>
<summary><strong>Python: load HelpSteer2 once and create a leakage-resistant prompt split</strong></summary>

```python
import copy
import json
import math
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)


def seed_everything(seed=1818):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


data_path = Path("assets/dl18-helpsteer2-mini.jsonl")
if not data_path.exists():
    data_path = Path("ipynb/Deep-Learning/assets/dl18-helpsteer2-mini.jsonl")

rows = [json.loads(line) for line in data_path.read_text(encoding="utf-8").splitlines()]
grouped = defaultdict(list)
for row in rows:
    grouped[row["pair_id"]].append(row)


def human_score(row):
    # Verbosity is audited separately rather than rewarded by default.
    return (
        0.4 * row["helpfulness"]
        + 0.3 * row["correctness"]
        + 0.2 * row["coherence"]
        + 0.1 * row["complexity"]
    )


pairs = []
for pair_id, candidates in grouped.items():
    assert len(candidates) == 2 and candidates[0]["prompt"] == candidates[1]["prompt"]
    ordered = sorted(candidates, key=human_score, reverse=True)
    if human_score(ordered[0]) == human_score(ordered[1]):
        continue
    pairs.append({"pair_id": pair_id, "prompt": ordered[0]["prompt"], "chosen": ordered[0], "rejected": ordered[1]})

seed_everything()
random.Random(1818).shuffle(pairs)
train_pairs, val_pairs, test_pairs = pairs[:168], pairs[168:204], pairs[204:]
train_ids = {pair["pair_id"] for pair in train_pairs}
test_ids = {pair["pair_id"] for pair in test_pairs}

assert len(pairs) == 240 and len(train_pairs) == 168 and len(val_pairs) == len(test_pairs) == 36
assert train_ids.isdisjoint(test_ids)
assert all(human_score(pair["chosen"]) > human_score(pair["rejected"]) for pair in pairs)
print({"prompt split": (168, 36, 36), "responses": len(rows), "license": "CC BY 4.0"})
```

</details>

The **prompt**, not the response row, is the independent split unit. Otherwise two answers to the same instruction could appear on opposite sides of the boundary, allowing a model or vectorizer to recognize test prompts during training.

### **What Does Post-Training Change?** {#what-post-training-changes}

A base language model estimates $p_{	heta_0}(y\mid x)$ from broad text. Deployment asks for a narrower conditional behavior: answer the user's instruction, satisfy a policy, preserve capabilities, and avoid pathological strategies that merely look good to an evaluator. Post-training therefore changes the **conditional distribution over responses**, not the factual contents of a database. It can make desired responses more probable, but it cannot reliably install knowledge absent from the model or data.

A useful abstraction is regularized policy optimization:

$$
\max_\theta\;\mathbb E_{x\sim\mathcal D,\,y\sim\pi_\theta(\cdot|x)}[R(x,y)]
-\beta\,\mathbb E_x[D_{\mathrm{KL}}(\pi_\theta(\cdot|x)\|\pi_{\mathrm{ref}}(\cdot|x))].
$$

Here $R$ is an imperfect operational objective, $\pi_{\mathrm{ref}}$ is usually the SFT policy, and $\beta$ prices movement away from that reference. A small $\beta$ permits aggressive optimization and more reward exploitation; a large $\beta$ preserves the reference but may prevent useful adaptation. SFT implements the first behavioral move with demonstrations, while RLHF and direct preference methods estimate or optimize relative preferences.

Alignment is not equivalent to “high reward.” The target is multi-dimensional: instruction following, factuality, calibration, safety, style, robustness, privacy, and operational cost can conflict. The model, reward model, data collection process, and evaluation suite form one coupled system. Improving a scalar proxy while ignoring this system is precisely how reward hacking arises.

| Stage | Main supervision | What is directly optimized | Characteristic limitation |
|---|---|---|---|
| Pretraining | raw token sequences | next-token likelihood | deployment intent is implicit |
| SFT | selected demonstrations | likelihood of target responses | imitates annotation distribution |
| Reward modeling | response comparisons | preference-order likelihood | evaluator bias and misspecification |
| RLHF / PPO | model rollouts and learned reward | regularized expected reward | costly and optimization-sensitive |
| DPO-family methods | fixed preference pairs | relative policy log-ratio | limited by offline pair coverage |
| Evaluation and red teaming | held-out tests and human review | diagnosis, not one training loss | never exhaustive |

### **Supervised Fine-Tuning** {#supervised-fine-tuning}

Supervised fine-tuning (SFT) teaches an autoregressive model to reproduce high-quality responses. For a prompt $x$ and demonstration $y=(y_1,\ldots,y_T)$, response-only SFT minimizes

$$
\mathcal L_{\mathrm{SFT}}(\theta)
=-\sum_{t=1}^{T}m_t\log \pi_\theta(y_t\mid x,y_{<t}),
$$

where $m_t=1$ for response targets and $m_t=0$ for prompt tokens. The prompt remains visible as context, but its tokens usually receive the ignore index (`-100` in PyTorch). Training on prompt tokens can waste capacity reproducing the user's text and makes examples with long prompts dominate the loss.

![Response-only SFT exposes prompt tokens as context while masking them from the loss.](assets/dl18-sft-mask.svg){fig-align="center" width="76%" fig-alt="Prompt and separator tokens have labels minus one hundred, while response tokens retain next-token labels and contribute gradients."}

Packing several examples into one sequence improves accelerator utilization, but attention and labels must not cross example boundaries. Losses should also be normalized by the intended unit: token averaging weights long answers more heavily, whereas per-example normalization gives each instruction comparable influence.

<details>
<summary><strong>PyTorch: construct response-only labels from a HelpSteer2 demonstration</strong></summary>

```python
BOS, SEP, EOS, VOCAB_SIZE = 256, 257, 258, 259


def encode_sft_example(prompt, response, prompt_limit=96, response_limit=160):
    prompt_bytes = list(prompt.encode("utf-8")[:prompt_limit])
    response_bytes = list(response.encode("utf-8")[:response_limit])
    full = [BOS] + prompt_bytes + [SEP] + response_bytes + [EOS]
    input_ids = torch.tensor(full[:-1], dtype=torch.long)
    labels = torch.tensor(full[1:], dtype=torch.long)
    separator_index = 1 + len(prompt_bytes)
    labels[:separator_index] = -100
    return input_ids, labels, separator_index


example = train_pairs[0]
input_ids, labels, separator_index = encode_sft_example(example["prompt"], example["chosen"]["response"])
seed_everything(1819)
logits = torch.randn(len(input_ids), VOCAB_SIZE, requires_grad=True)
loss = F.cross_entropy(logits, labels, ignore_index=-100)
loss.backward()

ignored_gradient = logits.grad[:separator_index].abs().sum()
response_gradient = logits.grad[separator_index:].abs().sum()
assert ignored_gradient == 0 and response_gradient > 0
print({"input tokens": len(input_ids), "masked targets": separator_index, "response NLL": round(float(loss.detach()), 3)})
```

</details>

SFT is stable and indispensable, but it only learns from one selected response per prompt. It does not directly express that another fluent answer was subtly worse. Common failure modes include template overfitting, verbosity inherited from demonstrations, catastrophic capability loss from excessive learning rates, and false confidence when the demonstrations omit uncertainty language.

### **Instruction Data Construction** {#instruction-data-construction}

Instruction data construction is part of the algorithm. A demonstration is not “just text”; it encodes a task distribution, rubric, author population, policy boundary, and quality-control process. A strong pipeline records provenance and license, filters personal or unsafe material, deduplicates semantically similar prompts, defines what annotators should do when uncertain, and reserves adversarial and out-of-domain tests before training begins.

![A data funnel narrows raw responses through provenance, quality, and split checks.](assets/dl18-data-funnel.svg){fig-align="center" width="70%" fig-alt="A four-stage funnel applies source checks, privacy and safety review, rubric and deduplication audits, then prompt-level train validation and test splitting."}

Dataset balance must be interpreted at several levels. Counting prompts can hide that one domain contains much longer answers and therefore more loss-bearing tokens. Average ratings can hide a small group with severe correctness failures. Near-duplicate prompts can inflate held-out performance. For preference data, tiny score differences also create weak or unstable labels; ties are better retained explicitly or removed according to a declared policy than arbitrarily broken.

<details>
<summary><strong>Python: audit lengths, dimensions, margins, and split provenance</strong></summary>

```python
def split_audit(selected_pairs):
    response_lengths = [len(pair[key]["response"]) for pair in selected_pairs for key in ("chosen", "rejected")]
    score_margins = [human_score(pair["chosen"]) - human_score(pair["rejected"]) for pair in selected_pairs]
    attribute_means = {
        attribute: round(float(np.mean([pair[key][attribute] for pair in selected_pairs for key in ("chosen", "rejected")])), 3)
        for attribute in ("helpfulness", "correctness", "coherence", "complexity", "verbosity")
    }
    return {
        "prompts": len(selected_pairs),
        "median response chars": int(np.median(response_lengths)),
        "p95 response chars": int(np.percentile(response_lengths, 95)),
        "median preference margin": round(float(np.median(score_margins)), 3),
        "attribute means": attribute_means,
    }


audits = {"train": split_audit(train_pairs), "validation": split_audit(val_pairs), "test": split_audit(test_pairs)}
assert {pair["prompt"] for pair in train_pairs}.isdisjoint({pair["prompt"] for pair in test_pairs})
assert all(audits[name]["p95 response chars"] <= 2200 for name in audits)
print(audits)
```

</details>

Filtering this teaching subset to moderate response lengths keeps CPU examples practical and changes the population being studied. The chapter therefore reports it as a mechanism dataset, not as an unbiased estimate of full HelpSteer2 or real assistant traffic.

### **Preference Data and Pairwise Feedback** {#preference-data-pairwise-feedback}

Pairwise feedback asks which response better satisfies a rubric for the same prompt. It is often easier than assigning a globally calibrated score because the annotator only compares two concrete candidates. A record has the form $(x,y_w,y_l)$, where $y_w$ is preferred and $y_l$ rejected.

![Pairwise feedback compares two responses under one explicit rubric.](assets/dl18-pairwise-feedback.svg){fig-align="center" width="74%" fig-alt="One prompt branches to a preferred and rejected response, which are compared under a rubric covering helpfulness, correctness, coherence, and uncertainty."}

Preferences depend on candidate generation. If both candidates come from nearly identical policies, differences may be too subtle; if one is obviously broken, labels are easy but teach little about the current decision boundary. Candidate order should be randomized, annotator identity and uncertainty should be recorded, and ties should be permitted. Position bias, style bias, demographic mismatch, and rubric ambiguity are data-generating mechanisms, not random noise that scale automatically removes.

HelpSteer2 provides separate scalar dimensions rather than a single universal preference. This chapter constructs a transparent composite for teaching, while keeping verbosity outside the target so it can be audited as a possible shortcut.

<details>
<summary><strong>Python: measure how a one-dimensional judge disagrees with a multi-attribute preference</strong></summary>

```python
def proxy_judge(pair):
    # A deliberately narrow judge sees helpfulness only.
    delta = pair["chosen"]["helpfulness"] - pair["rejected"]["helpfulness"]
    if delta == 0:
        return "tie"
    return "chosen" if delta > 0 else "rejected"


proxy_labels = [proxy_judge(pair) for pair in test_pairs]
agreement = sum(label == "chosen" for label in proxy_labels) / len(proxy_labels)
tie_rate = sum(label == "tie" for label in proxy_labels) / len(proxy_labels)
assert 0 <= agreement <= 1 and 0 <= tie_rate <= 1
print({"helpfulness-only agreement": round(agreement, 3), "helpfulness ties": round(tie_rate, 3), "test pairs": len(test_pairs)})
```

</details>

Disagreement does not prove that either label is wrong: the composite itself embeds weights. The diagnostic shows why an alignment report must publish its rubric and preserve the underlying dimensions rather than presenting “human preference” as a context-free fact.

### **Reward Modeling** {#reward-modeling}

A reward model maps a prompt-response pair to a scalar $r_\phi(x,y)$. In the Bradley-Terry model,

$$
P_\phi(y_w\succ y_l\mid x)=\sigma\!\left(r_\phi(x,y_w)-r_\phi(x,y_l)\right),
$$

so the pairwise negative log-likelihood is

$$
\mathcal L_{\mathrm{RM}}(\phi)=-\mathbb E_{(x,y_w,y_l)}\log\sigma(r_w-r_l)
=\mathbb E\,\operatorname{softplus}(-(r_w-r_l)).
$$

![A Bradley-Terry reward model turns score differences into preference probabilities.](assets/dl18-reward-model.svg){fig-align="center" width="74%" fig-alt="Prompt-response features pass through a scalar reward head, and the sigmoid of the chosen minus rejected reward gives the probability of preference."}

Only the difference $r_w-r_l$ is identified: adding the same constant to every score changes nothing. Reward values are therefore not absolute units of human utility. Calibration, subgroup performance, score drift, and out-of-distribution behavior matter alongside pairwise accuracy.

<details>
<summary><strong>PyTorch: train a compact pairwise reward model on fixed HelpSteer2 features</strong></summary>

```python
def candidate_text(pair, key):
    return pair["prompt"] + "\n<response>\n" + pair[key]["response"]


# Learned vocabulary and IDF statistics are fit on training text only.
vectorizer = TfidfVectorizer(max_features=1800, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
vectorizer.fit([candidate_text(pair, key) for pair in train_pairs for key in ("chosen", "rejected")])


def pair_features(selected_pairs):
    chosen = vectorizer.transform([candidate_text(pair, "chosen") for pair in selected_pairs]).toarray()
    rejected = vectorizer.transform([candidate_text(pair, "rejected") for pair in selected_pairs]).toarray()
    return torch.tensor(np.stack([chosen, rejected], axis=1), dtype=torch.float32)


train_x, val_x, test_x = map(pair_features, (train_pairs, val_pairs, test_pairs))
seed_everything(1820)
reward_model = nn.Linear(train_x.shape[-1], 1)
reward_optimizer = torch.optim.AdamW(reward_model.parameters(), lr=0.04, weight_decay=2e-3)
reward_losses = []
for _ in range(350):
    scores = reward_model(train_x).squeeze(-1)
    loss = F.softplus(-(scores[:, 0] - scores[:, 1])).mean()
    reward_optimizer.zero_grad()
    loss.backward()
    reward_optimizer.step()
    reward_losses.append(float(loss.detach()))


def reward_metrics(features):
    with torch.no_grad():
        scores = reward_model(features).squeeze(-1)
        margins = scores[:, 0] - scores[:, 1]
    return {"pair accuracy": float((margins > 0).float().mean()), "mean margin": float(margins.mean())}


rm_validation = reward_metrics(val_x)
rm_test = reward_metrics(test_x)
assert reward_losses[-1] < reward_losses[0] and 0 <= rm_test["pair accuracy"] <= 1
print({"features": train_x.shape[-1], "loss": (round(reward_losses[0], 3), round(reward_losses[-1], 3)), "validation": {k: round(v, 3) for k, v in rm_validation.items()}, "test": {k: round(v, 3) for k, v in rm_test.items()}})
```

</details>

This linear TF-IDF model intentionally exposes the objective; production reward models normally reuse a pretrained transformer and attach a scalar head. The small held-out set makes accuracy noisy, and lexical features are easy to exploit. Those limitations are useful here because later sections can observe proxy optimization rather than assuming the evaluator is correct.

### **Reinforcement Learning from Human Feedback** {#reinforcement-learning-human-feedback}

RLHF separates preference measurement from policy improvement. Humans compare responses; a reward model learns the comparison rule; the policy then generates new responses and is optimized against that learned reward. In the InstructGPT-style pipeline, SFT provides both a strong initialization and the frozen reference policy used to limit drift.

![RLHF alternates policy rollouts, reward evaluation, and KL-regularized PPO updates.](assets/dl18-rlhf-loop.svg){fig-align="center" width="76%" fig-alt="Prompts enter a policy, complete responses receive reward and KL penalties, PPO updates the policy, and a frozen reference anchors the update."}

A common shaped sequence reward is

$$
\widetilde R(x,y)=r_\phi(x,y)-\beta\log\frac{\pi_\theta(y|x)}{\pi_{\mathrm{ref}}(y|x)},
\qquad
\log\pi_\theta(y|x)=\sum_t\log\pi_\theta(y_t|x,y_{<t}).
$$

The first term scores the completed response. The second penalizes a sequence that became much more likely under the updated policy than under the reference. Implementations often distribute token-level KL penalties across the trajectory and train a value head to assign terminal reward backward through tokens.

RLHF is **on-policy with respect to the current generator**: after enough policy movement, old responses no longer characterize what the model now emits. This adaptivity can expose new failure modes and is one advantage over a fixed preference dataset. It also makes RLHF expensive because generation dominates training, reward and policy versions must be synchronized, and rollout data becomes stale quickly.

A reliable system logs policy and reference log probabilities, raw reward, KL penalty, response length, entropy, clipping fraction, value error, and held-out human judgments. Rising training reward with collapsing entropy or rapidly increasing KL is a warning, not success.

### **PPO for Model Alignment** {#ppo-model-alignment}

Chapter 17 introduced PPO as a general on-policy actor-critic method. In language alignment, a state is the prompt plus generated prefix, an action is the next token, and an episode is a complete response. PPO reuses a rollout for several gradient epochs while clipping the probability ratio

$$
\rho_t(\theta)=\frac{\pi_\theta(y_t|x,y_{<t})}{\pi_{\mathrm{old}}(y_t|x,y_{<t})},
\qquad
L^{\mathrm{clip}}=\mathbb E_t\left[\min(\rho_t\hat A_t,\operatorname{clip}(\rho_t,1-\epsilon,1+\epsilon)\hat A_t)\right].
$$

![Language-model PPO maps token actions to a terminal response score and uses a critic plus KL shaping for credit assignment.](assets/dl18-ppo-alignment.svg){fig-align="center" width="74%" fig-alt="Token probabilities generate a response, a terminal reward is assigned backward with a critic and KL shaping, and PPO clips probability-ratio incentives."}

PPO clipping constrains the sampled surrogate incentive, not the actual KL everywhere. The explicit reference KL and a controller that adjusts $\beta$ remain important. Padding masks, EOS handling, reward whitening, advantage normalization, old-log-prob detachment, and the distinction between truncation and a genuine EOS are frequent sources of silent bugs.

<details>
<summary><strong>PyTorch: a two-response contextual-bandit abstraction of KL-regularized PPO</strong></summary>

```python
class CandidatePolicy(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        self.scorer = nn.Linear(feature_dim, 1)

    def forward(self, features):
        return self.scorer(features).squeeze(-1)


def categorical_kl(logits, reference_logits):
    log_p = logits.log_softmax(-1)
    log_q = reference_logits.log_softmax(-1)
    return (log_p.exp() * (log_p - log_q)).sum(-1)


seed_everything(1821)
reference_policy = CandidatePolicy(train_x.shape[-1])
sft_optimizer = torch.optim.AdamW(reference_policy.parameters(), lr=0.025, weight_decay=1e-3)
chosen_targets = torch.zeros(len(train_x), dtype=torch.long)
for _ in range(180):
    sft_loss = F.cross_entropy(reference_policy(train_x), chosen_targets)
    sft_optimizer.zero_grad(); sft_loss.backward(); sft_optimizer.step()

for parameter in reference_policy.parameters():
    parameter.requires_grad_(False)
ppo_policy = copy.deepcopy(reference_policy)
for parameter in ppo_policy.parameters():
    parameter.requires_grad_(True)
ppo_optimizer = torch.optim.AdamW(ppo_policy.parameters(), lr=0.006, weight_decay=1e-3)

with torch.no_grad():
    train_rewards = reward_model(train_x).squeeze(-1)
    train_rewards = (train_rewards - train_rewards.mean()) / (train_rewards.std() + 1e-6)
    reference_train_logits = reference_policy(train_x)

beta, epsilon = 0.08, 0.2
ppo_log = []
for update in range(24):
    old_policy = copy.deepcopy(ppo_policy).eval()
    with torch.no_grad():
        old_logits = old_policy(train_x)
        old_distribution = torch.distributions.Categorical(logits=old_logits)
        actions = old_distribution.sample()
        old_log_prob = old_distribution.log_prob(actions)
        baseline = (old_logits.softmax(-1) * train_rewards).sum(-1)
        advantages = train_rewards.gather(1, actions[:, None]).squeeze(1) - baseline
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-6)
    for _ in range(4):
        logits = ppo_policy(train_x)
        distribution = torch.distributions.Categorical(logits=logits)
        ratio = (distribution.log_prob(actions) - old_log_prob).exp()
        surrogate = torch.minimum(ratio * advantages, ratio.clamp(1 - epsilon, 1 + epsilon) * advantages)
        kl = categorical_kl(logits, reference_train_logits).mean()
        loss = -surrogate.mean() + beta * kl - 0.002 * distribution.entropy().mean()
        ppo_optimizer.zero_grad(); loss.backward(); ppo_optimizer.step()
    ppo_log.append((float(kl.detach()), float(((ratio - 1).abs() > epsilon).float().mean())))

with torch.no_grad():
    reference_test_logits = reference_policy(test_x)
    ppo_test_logits = ppo_policy(test_x)
    ppo_preferred_probability = float(ppo_test_logits.softmax(-1)[:, 0].mean())
    ppo_test_kl = float(categorical_kl(ppo_test_logits, reference_test_logits).mean())
assert math.isfinite(ppo_test_kl) and 0 <= ppo_preferred_probability <= 1
print({"preferred probability": round(ppo_preferred_probability, 3), "test KL to reference": round(ppo_test_kl, 4), "last train clip fraction": round(ppo_log[-1][1], 3)})
```

</details>

The code treats the two recorded responses as the complete action set. It preserves PPO's sampling, old-policy ratio, clipping, baseline, and reference KL, but it is not token-level RLHF and cannot expose generation-time pathologies. Its purpose is to isolate the update geometry on the same data used by SFT and the reward model.

### **Direct Preference Optimization** {#direct-preference-optimization}

DPO removes the explicit reward-model-plus-RL loop from training. Under a KL-regularized reward formulation, the optimal policy implies a reward proportional to the log-ratio between the target and reference policies. Substituting that relation into the Bradley-Terry preference model yields

$$
\mathcal L_{\mathrm{DPO}}(\theta)
=-\mathbb E\log\sigma\left(\beta\left[
\log\frac{\pi_\theta(y_w|x)}{\pi_{\mathrm{ref}}(y_w|x)}
-\log\frac{\pi_\theta(y_l|x)}{\pi_{\mathrm{ref}}(y_l|x)}
\right]\right).
$$

![DPO optimizes the chosen response's policy improvement relative to the rejected response and a reference policy.](assets/dl18-dpo-log-ratio.svg){fig-align="center" width="72%" fig-alt="Chosen and rejected log policy ratios are subtracted, scaled by beta, and passed through negative log sigmoid."}

The bracketed term is a **difference of relative improvements**. DPO does not merely maximize chosen likelihood: it asks the current policy to favor $y_w$ over $y_l$ more strongly than the reference did. The inverse-temperature $\beta$ sets the scale. Its convention varies across implementations, so equations and library settings must be checked together.

<details>
<summary><strong>PyTorch: optimize DPO on the same HelpSteer2 preference pairs</strong></summary>

```python
dpo_policy = copy.deepcopy(reference_policy)
for parameter in dpo_policy.parameters():
    parameter.requires_grad_(True)
dpo_optimizer = torch.optim.AdamW(dpo_policy.parameters(), lr=0.008, weight_decay=1e-3)
with torch.no_grad():
    reference_train_log_probs = reference_policy(train_x).log_softmax(-1)

dpo_beta = 0.2
dpo_losses = []
for _ in range(220):
    policy_log_probs = dpo_policy(train_x).log_softmax(-1)
    policy_log_ratio = policy_log_probs[:, 0] - policy_log_probs[:, 1]
    reference_log_ratio = reference_train_log_probs[:, 0] - reference_train_log_probs[:, 1]
    dpo_loss = F.softplus(-dpo_beta * (policy_log_ratio - reference_log_ratio)).mean()
    dpo_optimizer.zero_grad(); dpo_loss.backward(); dpo_optimizer.step()
    dpo_losses.append(float(dpo_loss.detach()))

with torch.no_grad():
    dpo_test_logits = dpo_policy(test_x)
    dpo_preferred_probability = float(dpo_test_logits.softmax(-1)[:, 0].mean())
    dpo_test_kl = float(categorical_kl(dpo_test_logits, reference_test_logits).mean())
assert dpo_losses[-1] < dpo_losses[0] and dpo_test_kl >= 0
print({"loss": (round(dpo_losses[0], 3), round(dpo_losses[-1], 3)), "preferred probability": round(dpo_preferred_probability, 3), "test KL to reference": round(dpo_test_kl, 4)})
```

</details>

DPO is operationally simpler: no online rollout service, learned critic, reward-model serving, or PPO loop is required. That simplicity does not make it immune to overfitting, label noise, length bias, or out-of-distribution prompts. It is fundamentally offline unless new preference pairs are collected. Variants such as IPO, ORPO, KTO, and robust DPO alter assumptions or losses, but the central audit remains the same: data coverage, reference choice, regularization strength, and held-out behavior.

### **Rejection Sampling and Best-of-N** {#rejection-sampling-best-of-n}

Best-of-$N$ samples $N$ responses from a proposal policy, scores them, and returns the highest-scoring candidate. Rejection sampling fine-tuning can then add selected candidates back to the supervised dataset. These methods move optimization partly to inference and data selection rather than changing every token probability with policy gradients.

![Best-of-N generates several candidates and returns the reward model's highest-scoring response.](assets/dl18-best-of-n.svg){fig-align="center" width="72%" fig-alt="One prompt produces N candidates, a reward model scores them, and an argmax stage returns one response."}

If candidates were independent with quality CDF $F$, the maximum has CDF $F_{\max}(q)=F(q)^N$. Larger $N$ improves the expected maximum under the scorer, but costs roughly $N$ times the generation work before batching and cache reuse. Gains saturate when samples lack diversity. More importantly, selecting extreme proxy scores explores regions where reward-model error can dominate.

<details>
<summary><strong>Python: compare Best-of-N proxy reward and held-out human score</strong></summary>

```python
def pair_human_scores(selected_pairs):
    return torch.tensor([[human_score(pair["chosen"]), human_score(pair["rejected"])] for pair in selected_pairs], dtype=torch.float32)


with torch.no_grad():
    proposal_probs = reference_test_logits.softmax(-1)
    test_reward_scores = reward_model(test_x).squeeze(-1)
    test_human_scores = pair_human_scores(test_pairs)


def best_of_n_metrics(n, trials=250):
    generator = torch.Generator().manual_seed(1800 + n)
    proxy_values, human_values = [], []
    for _ in range(trials):
        samples = torch.multinomial(proposal_probs, n, replacement=True, generator=generator)
        sampled_rewards = test_reward_scores.gather(1, samples)
        winner_position = sampled_rewards.argmax(1, keepdim=True)
        winner = samples.gather(1, winner_position).squeeze(1)
        proxy_values.append(test_reward_scores.gather(1, winner[:, None]).mean())
        human_values.append(test_human_scores.gather(1, winner[:, None]).mean())
    return float(torch.stack(proxy_values).mean()), float(torch.stack(human_values).mean())


best_of_n_results = {n: tuple(round(value, 3) for value in best_of_n_metrics(n)) for n in (1, 2, 4, 8)}
assert best_of_n_results[8][0] >= best_of_n_results[1][0] - 1e-6
print({"N: (RM score, human composite)": best_of_n_results})
```

</details>

With only two recorded responses, this experiment isolates selection pressure rather than realistic language diversity. The guaranteed statement concerns the selected **reward-model score**, not true human utility. A production study should report pass@$N$, distinctness, latency, token cost, evaluator uncertainty, and human re-evaluation of high-scoring samples.

### **AI Feedback and Scalable Oversight** {#ai-feedback-scalable-oversight}

Human comparison is expensive and difficult to scale to long reasoning traces or specialist domains. Reinforcement learning from AI feedback (RLAIF) uses a model judge to critique, rank, or revise responses under an explicit rubric. [Constitutional AI](https://arxiv.org/abs/2212.08073) separates a supervised critique-and-revision phase from an AI-feedback preference phase.

![Constitutional AI uses an explicit rubric to critique and revise responses before updating policies or judges.](assets/dl18-rlaif.svg){fig-align="center" width="74%" fig-alt="A draft response is checked against a constitution, converted into a revision or preference, and used for an update with human audits in the loop."}

A scalable loop can ask several judges, randomize response order, require evidence, permit abstention, and route high-uncertainty cases to humans. This scales a **specified rubric**; it does not prove the rubric is complete. Model judges can favor their own style, be sensitive to position and verbosity, share blind spots with the policy, or be manipulated by text inside the response.

The helpfulness-only diagnostic earlier is a minimal analogue of judge misspecification. A more realistic evaluation compares AI and human labels on a sequestered audit set and stratifies agreement by domain, language, response length, safety category, and judge confidence. Agreement on easy pairs can coexist with systematic failure on exactly the cases where oversight is most valuable.

Scalable oversight additionally studies tasks whose outcomes are hard for humans to evaluate directly. Decomposition, debate, recursive critique, process supervision, and tool-assisted verification attempt to make latent errors observable. They change the evidence available to the evaluator; none removes the need for threat models and independent audits.

### **Online vs Offline Preference Learning** {#online-offline-preference-learning}

Offline preference learning trains on a fixed dataset collected from earlier policies. It is reproducible, easy to cache, and compatible with DPO. Its support is fixed: after the policy changes, the dataset may contain few examples resembling the new outputs. Online learning repeatedly samples the current policy, obtains new labels, updates, and audits again.

![Offline learning inherits fixed coverage, whereas online learning forms a policy-data feedback loop.](assets/dl18-online-offline.svg){fig-align="center" width="73%" fig-alt="An offline panel shows fixed prompts and pairs; an online loop sends the current policy to new feedback and back into an update."}

Online does not automatically mean better. Fresh labels are costly, the data distribution moves, regressions can contaminate later collection, and comparisons across model versions become harder. A practical hybrid begins with broad offline SFT and preferences, then spends online labeling budget on current-policy failures, uncertain reward-model comparisons, and high-risk slices.

<details>
<summary><strong>Python: use reward-model uncertainty to prioritize a feedback batch</strong></summary>

```python
with torch.no_grad():
    validation_reward_margin = (reward_model(val_x).squeeze(-1)[:, 0] - reward_model(val_x).squeeze(-1)[:, 1]).abs()
true_validation_margin = torch.tensor([
    human_score(pair["chosen"]) - human_score(pair["rejected"]) for pair in val_pairs
])
budget = 10
uncertain_ids = validation_reward_margin.argsort()[:budget]
random_ids = torch.randperm(len(val_pairs), generator=torch.Generator().manual_seed(1825))[:budget]
uncertainty_report = {
    "uncertainty-selected RM margin": float(validation_reward_margin[uncertain_ids].mean()),
    "random RM margin": float(validation_reward_margin[random_ids].mean()),
    "uncertainty-selected human margin": float(true_validation_margin[uncertain_ids].mean()),
    "random human margin": float(true_validation_margin[random_ids].mean()),
}
assert uncertainty_report["uncertainty-selected RM margin"] <= uncertainty_report["random RM margin"]
print({key: round(value, 3) for key, value in uncertainty_report.items()})
```

</details>

Small predicted margin identifies cases the reward model finds ambiguous, not necessarily cases humans find ambiguous. Combining uncertainty with domain coverage, safety risk, novelty, and disagreement across independently trained reward models is more robust than one acquisition score.

### **Reward Hacking and Over-Optimization** {#reward-hacking-over-optimization}

Reward hacking occurs when a policy finds behavior that scores highly under the proxy without satisfying the intended objective. The issue is a form of Goodhart's law: once a noisy measurement becomes an optimization target, the optimizer seeks regions where measurement error is favorable. [Scaling Laws for Reward Model Overoptimization](https://arxiv.org/abs/2210.10760) empirically separates proxy reward from held-out “gold” reward as optimization increases.

![Proxy reward can keep increasing after held-out human utility peaks.](assets/dl18-reward-overoptimization.svg){fig-align="center" width="73%" fig-alt="A blue proxy-reward curve rises with optimization pressure while a red held-out human utility curve rises, peaks, and then falls."}

Shortcuts include verbosity, confident tone, copied rubric language, refusal everywhere, reward-model-specific phrases, and sycophancy. KL regularization limits policy distance but cannot guarantee safety: a nearby policy can still exploit a local reward defect, while an overly strong KL can preserve undesirable reference behavior.

<details>
<summary><strong>Python: stress-test a deliberately misspecified length-sensitive proxy</strong></summary>

```python
with torch.no_grad():
    base_proxy = reward_model(test_x).squeeze(-1)
response_lengths = torch.tensor([
    [len(pair["chosen"]["response"]), len(pair["rejected"]["response"])] for pair in test_pairs
], dtype=torch.float32)
standardized_length = (response_lengths - response_lengths.mean()) / (response_lengths.std() + 1e-6)
true_scores = pair_human_scores(test_pairs)

reward_hacking_report = {}
for length_bonus in (0.0, 0.5, 1.0, 2.0):
    misspecified_proxy = base_proxy + length_bonus * standardized_length
    selected = misspecified_proxy.argmax(1, keepdim=True)
    reward_hacking_report[length_bonus] = {
        "optimized proxy": round(float(misspecified_proxy.gather(1, selected).mean()), 3),
        "human composite": round(float(true_scores.gather(1, selected).mean()), 3),
        "response chars": round(float(response_lengths.gather(1, selected).mean()), 1),
    }

assert reward_hacking_report[2.0]["response chars"] >= reward_hacking_report[0.0]["response chars"]
print(reward_hacking_report)
```

</details>

This is an explicit stress test, not a claim that HelpSteer2 rewards length. The point is to perturb the evaluator, optimize against it, and compare with held-out dimensions. Defenses include reward ensembles, adversarial data, uncertainty-aware penalties, early stopping, reference constraints, independent judges, and human review of extreme-score outputs. None substitutes for monitoring the actual deployed distribution.

### **Alignment Tax, Evaluation, and Safety** {#alignment-tax-evaluation-safety}

An **alignment tax** is a loss on some pre-existing capability, diversity, latency, or cost metric caused by post-training constraints. It is not inevitable and should not be inferred from one aggregate benchmark. SFT can improve task quality by clarifying intent; preference optimization can reduce useful diversity; Best-of-$N$ can improve selected quality while multiplying inference cost. The correct object is a Pareto surface, not one score.

![Alignment evaluation tracks task quality, preference fit, drift, safety, robustness, efficiency, and oversight separately.](assets/dl18-alignment-evaluation.svg){fig-align="center" width="76%" fig-alt="Seven panels form an evaluation ledger for task quality, preference fit, policy drift, safety, robustness, efficiency, and human oversight."}

<details>
<summary><strong>PyTorch: compare reference, PPO, and DPO on a shared evaluation ledger</strong></summary>

```python
test_lengths = response_lengths
test_human = true_scores
with torch.no_grad():
    test_rm = reward_model(test_x).squeeze(-1)


def policy_ledger(name, logits):
    probabilities = logits.softmax(-1)
    return {
        "policy": name,
        "preferred probability": float(probabilities[:, 0].mean()),
        "expected human composite": float((probabilities * test_human).sum(-1).mean()),
        "expected RM score": float((probabilities * test_rm).sum(-1).mean()),
        "KL to reference": float(categorical_kl(logits, reference_test_logits).mean()),
        "expected response chars": float((probabilities * test_lengths).sum(-1).mean()),
    }


ledgers = [
    policy_ledger("SFT reference", reference_test_logits),
    policy_ledger("PPO abstraction", ppo_test_logits),
    policy_ledger("DPO", dpo_test_logits),
]
for ledger in ledgers:
    print({key: (round(value, 3) if isinstance(value, float) else value) for key, value in ledger.items()})
assert all(math.isfinite(ledger["expected human composite"]) for ledger in ledgers)
```

</details>

The deterministic teaching run deliberately permits an uncomfortable result: the compact reward model can nearly separate its training pairs while remaining close to chance on held-out ordering, and an optimizer can raise expected learned reward without raising the held-out human composite. That is not a ranking of PPO against DPO. It is evidence that evaluator generalization must be established before stronger policy optimization is trusted.

The ledger is intentionally incomplete: HelpSteer2's five ratings cannot certify toxicity, privacy, bias, jailbreak resistance, calibration, or downstream factual correctness. Safety needs task-specific refusal tests, benign-overrefusal tests, multilingual and subgroup slices, adversarial prompts, tool-use constraints, and human escalation. Capability evaluation should include the unaligned base/SFT model to reveal regressions, while statistical reporting should include confidence intervals and multiple prompt templates.

Offline scores also miss system effects. Sampling temperature, system prompts, retrieval, tools, safety filters, quantization, and serving truncation can change deployment behavior. Evaluation artifacts must therefore record the exact model, tokenizer, decoding configuration, policy text, evaluator version, and date.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

| Method | Training signal | Online generation required? | Explicit reward model? | Main advantage | Main failure mode |
|---|---|---:|---:|---|---|
| SFT | target response tokens | no | no | stable, simple instruction learning | demonstration bias and no negative comparison |
| Reward modeling | chosen/rejected pairs | no | yes | reusable preference evaluator | proxy misspecification and shift |
| RLHF with PPO | current-policy rollouts | yes | usually yes | adapts to current policy outputs | system complexity and reward exploitation |
| DPO | fixed preference pairs | no | implicit | simple supervised-style optimization | offline coverage and reference sensitivity |
| Best-of-$N$ | sampled candidates and scorer | at inference | usually yes | no policy-gradient loop | inference cost and extreme-score bias |
| RLAIF | model critique or comparison | depends | judge or reward model | scalable rubric application | shared judge-policy blind spots |

A practical post-training workflow is:

1. Define the intended behavior and a multi-axis evaluation suite before optimizing.
2. Build provenance-aware demonstrations and split by prompt or conversation lineage.
3. Train SFT with response-only labels and monitor capability retention.
4. Collect randomized comparisons with ties, uncertainty, and annotator metadata.
5. Validate reward models by slice, calibration, and distribution shift, not training accuracy alone.
6. Choose DPO for a controlled offline baseline or PPO/RLHF when current-policy sampling justifies the operational cost.
7. Use Best-of-$N$ as an inference-time trade-off, and re-evaluate extreme scores with independent judgments.
8. Treat AI feedback as scalable rubric execution with human audits, not as ground truth.
9. Track reward, KL, entropy, response length, capability, safety, and cost together.
10. Stop or refresh data when proxy reward rises without held-out improvement.

Post-training is best understood as **measurement-aware policy shaping**. The algorithm matters, but its behavior is bounded by the demonstrations, comparisons, reference policy, evaluator, rollout distribution, and deployment tests surrounding it. The next chapter turns from objectives to the systems problem of training such models efficiently and at scale.